# 02 - Transformacao Silver Pix

Este notebook le a camada Bronze, padroniza colunas e tipos, aplica validacoes simples e salva a camada Silver.

## Camada Silver

A camada Silver representa dados tratados e preparados para analise. Nesta etapa sao padronizados nomes de colunas, referencia mensal, valores numericos e filtros de qualidade.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
import re
import unicodedata

from pyspark.sql import functions as F

from src.config import PIX_CLEAN_DIR, PIX_RAW_DIR, create_project_directories
from src.data_quality import ensure_columns, ensure_not_empty
from src.spark_session import get_spark_session

create_project_directories(verbose=False)
spark = get_spark_session("02-transform-silver-pix")

In [ ]:
bronze_df = spark.read.parquet(str(PIX_RAW_DIR))
ensure_not_empty(bronze_df, "Bronze Pix raw")
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

In [ ]:
def normalize_column_name(name: str) -> str:
    text = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^0-9a-zA-Z]+", "_", text).strip("_").lower()
    return text

normalized_df = bronze_df
for original_name in bronze_df.columns:
    normalized_df = normalized_df.withColumnRenamed(original_name, normalize_column_name(original_name))

print(normalized_df.columns)

In [ ]:
required_candidates = ["anomes", "valor", "quantidade"]
ensure_columns(normalized_df, required_candidates, "Silver Pix normalizada")

def parse_number(column_name: str):
    return F.regexp_replace(F.col(column_name).cast("string"), ",", ".").cast("double")

silver_df = (
    normalized_df
    .dropna(how="all")
    .withColumn("ano_mes", F.col("anomes").cast("string"))
    .withColumn("ano_mes", F.regexp_replace(F.col("ano_mes"), r"[^0-9]", ""))
    .withColumn("ano", F.substring("ano_mes", 1, 4).cast("int"))
    .withColumn("mes", F.substring("ano_mes", 5, 2).cast("int"))
    .withColumn("quantidade_transacoes", parse_number("quantidade"))
    .withColumn("valor_total", parse_number("valor"))
)

for optional_name in ["pag_regiao", "rec_regiao", "pag_pfpj", "rec_pfpj", "natureza", "finalidade", "formainiciacao"]:
    if optional_name not in silver_df.columns:
        silver_df = silver_df.withColumn(optional_name, F.lit(None).cast("string"))

silver_df = (
    silver_df
    .filter(F.col("ano_mes").rlike(r"^[0-9]{6}$"))
    .filter(F.col("quantidade_transacoes").isNotNull())
    .filter(F.col("valor_total").isNotNull())
    .filter(F.col("quantidade_transacoes") >= 0)
    .filter(F.col("valor_total") >= 0)
    .select(
        "ano_mes",
        "ano",
        "mes",
        "quantidade_transacoes",
        "valor_total",
        "pag_regiao",
        "rec_regiao",
        "pag_pfpj",
        "rec_pfpj",
        "natureza",
        "finalidade",
        "formainiciacao",
    )
)

ensure_not_empty(silver_df, "Silver Pix clean")

In [ ]:
silver_df.printSchema()
silver_df.show(10, truncate=False)
print(f"Registros finais na camada Silver: {silver_df.count()}")

In [ ]:
silver_df.write.mode("overwrite").parquet(str(PIX_CLEAN_DIR))
print(f"Camada Silver gravada em: {PIX_CLEAN_DIR.relative_to(PROJECT_DIR)}")

In [ ]:
spark.stop()